# Урок 13 - Памет на агента с когнитивни графи за знания


## Настройка

Този тетрадка демонстрира как да създадете интелигентен **кодиращ асистент** с постоянна памет, използвайки знанията на [**Cognee**](https://www.cognee.ai/) графи и **Microsoft Agent Framework** (MAF).

Cognee преобразува неструктуриран текст в структурирана, подлежаща на заявки граф от знания, подплатен с векторни вграждания — давайки на вашия агент богата, с осъзнаване на връзките дългосрочна памет.

### Какво ще научите
1. **Създаване на графи от знания**: Преобразувайте профили на разработчици и добри практики в структурирано, подлежащо на заявки знание.
2. **Интегриране на Cognee с MAF**: Използвайте функции `@tool`, за да позволите на MAF агент да заявява графа от знания на Cognee.
3. **Разговори със сесионна осведоменост**: Поддържайте контекст в множество въпроси в същата сесия.
4. **Дългосрочна памет**: Запазвайте важни знания през сесии и ги извличайте в нови разговори.

### Предварителни изисквания
- Python 3.9+
- Redis, работещ локално (`docker run -d -p 6379:6379 redis`) за управление на сесии
- Ключ за LLM API (например OpenAI) — задайте `LLM_API_KEY` в `.env`
- `CACHING=true` в `.env` (изисква се за сесиите на Cognee)
- Проект Azure AI Foundry с внедрен чат модел
- `AZURE_AI_PROJECT_ENDPOINT` и `AZURE_AI_MODEL_DEPLOYMENT_NAME` в `.env`
- Автентикация с Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Видове памет на агента

Този бележник изследва същите три типа памет от основния бележник на Урок 13, но използва Cognee като заден край за дългосрочна памет:

| Вид памет | Механизъм | Продължителност |
|---|---|---|
| **Работна** | `agent.create_session()` (MAF) | Един разговор |
| **Краткосрочна** | Кеш на сесия Cognee (Redis) | Една сесия |
| **Дългосрочна** | Знаниева графика на Cognee + вектори | Постоянна |

### Архитектура на паметта на Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Подгответе Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Част 1 — Създаване на база знания

Ние въвеждаме три вида данни, за да създадем изчерпателна база знания за нашия асистент по програмиране:

1. **Профил на разработчика** — личен опит и техническа подготовка  
2. **Най-добри практики в Python** — Зен на Python с практически насоки  
3. **Исторически разговори** — минали сесии с въпроси и отговори между разработчици и AI асистенти


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Визуализиране на графа на знанието

Cognee може да изобрази интерактивна HTML визуализация на извлечените обекти и връзки.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Обогатете паметта с Memify

`memify()` анализира графа на знанията и генерира интелигентни правила — идентифицирайки модели, най-добри практики и връзки между понятията.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Част 2 — MAF агент с Cognee инструменти

Сега създаваме MAF агент, който може да търси в графата с знания на Cognee чрез функции `@tool`. Това позволява на агента да използва пълната мощ на семантично търсене, осведомено за графа, като същевременно поддържа контекста на разговора чрез сесии.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Работна памет със сесии

`AgentSession` (създаден чрез `agent.create_session()`) осигурява работна памет в рамките на сесия. Агента може да се обръща към по-ранни съобщения, като същевременно запитва графа за дългосрочни знания на Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Нова сесия — Дългосрочната памет остава

Започването на нова сесия изчиства работната памет, но графът на знанието Cognee все още е наличен. Актьорът може да извлече същото дългосрочно знание в напълно нов разговор.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В този бележник създадохте кодиращ асистент, който комбинира **работната памет на MAF** (`agent.create_session()`) с **дългосрочния граф на знанието на Cognee**.

### Какво научихте
1. **Създаване на граф на знанието**: Cognee приема неструктуриран текст и изгражда граф + векторна памет.
2. **Обогатяване на графа с memify**: Производни факти и по-богати връзки върху съществуващия ви граф.
3. **Интеграция MAF + Cognee**: `@tool` функции позволяват на агент на MAF да прави заявки към графа на Cognee по естествен начин.
4. **Работна памет + дългосрочна памет**: `AgentSession` (чрез `agent.create_session()`) предоставя контекст на сесията, докато Cognee осигурява постоянни знания.
5. **Филтрирано търсене с NodeSets**: Насочване към конкретни подмножества от графа на знанието (например само принципи).

### Основни изводи
- **Cognee** превръща суровия текст в структурирана, с осъзнаване на отношения памет — по-мощна от плоско векторно хранилище.
- **`@tool` функциите** свързват агентите на MAF с външни системи за знания по чист начин.
- **`AgentSession`** (чрез `agent.create_session()`) запазва контекста на всеки разговор отделно от дългоживеещите знания.
- Същият граф на знанието обслужва множество сесии и агенти.

### Приложения в реалния свят
- **Помощници за разработчици**: Преглед на код, анализ на инциденти, асистенти за архитектура
- **Помощници, насочени към клиенти**: Агенти за поддръжка над продуктови документи, често задавани въпроси и бележки от CRM
- **Вътрешни експертни помощници**: Помощници по политика, правни или сигурност с умозаключение спрямо насоки
- **Единни нива на данни**: Комбиниране на структурирани и неструктурирани данни в един запитващ се граф

### Следващи стъпки
- Експериментиране с времево съзнание в Cognee
- Определяне на OWL онтология за специфично качество на домейна на графа
- Добавяне на цикли за обратна връзка от потребителя за подобряване на търсенето с времето
- Масштабиране към мултиагентни системи, споделящи един и същ паметов слой на Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:  
Този документ е преведен чрез AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стараем да осигурим точност, моля, имайте предвид, че автоматизираните преводи може да съдържат грешки или неточности. Оригиналният документ на неговия език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
